# Sınav Çizelgeleme (4 Hafta) 

# Proje Amacı ve Kapsamı
Bu çalışma; sınırlı zaman dilimleri ve derslik kapasiteleri göz önünde bulundurularak, akademik bir sınav/oturum takviminin optimize edilmesini hedeflemektedir. Süreç, mevcut kaynakların en verimli şekilde kullanılmasını ve çakışmaların önlenmesini temel alır.

# Gerekçe ve Kısıtlar
Kaynakların (zaman ve mekan) kısıtlı olması nedeniyle çizelgeleme sürecinde aşağıdaki parametreler dikkate alınmıştır:

* Zaman Çizelgesi: 4 hafta (Pazartesi-Cuma) toplam 20 iş gününü kapsamaktadır.

* Oturum Yapısı: Günlük 3 oturum (09:00, 13:00, 16:00) olmak üzere toplam 60 zaman dilimi mevcuttur.

* Derslik Yönetimi: Kapasite kurallarına tabi 5 farklı derslik tanımlanmıştır.

* Mekansal Kısıtlar: "Arena A" salonu Cuma günleri kullanıma kapalıdır.

* Özel Ekipman Gereksinimi: "PC Laboratuvarı" yalnızca bilgisayar kullanımı gerektiren (Requires_PC = Yes) sınavlara tahsis edilmiştir.

# Çıktı ve Raporlama
Oluşturulan nihai sınav takvimi şu verileri içermektedir:

* Grup Bilgisi: Sınava girecek bölüm, sınıf düzeyi ve şube bilgisi.

* Zaman Tahsisi: Atanan gün ve oturum saati.

* Mekan Tahsisi: Görevlendirilen derslik/salon bilgisi.

* Kapasite Analizi: Kontenjan ve doluluk oranlarının uygunluk denetimi.





In [1]:
#Bize gerekli olan kütüphaneleri içeriye aktarıyorum.
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


In [2]:
# Veri kontrolünü sağlıyorum
students = pd.read_csv("students.csv")
students.head()

,Student_ID,Name,Major,Year
0,S00001,Ogrenci_1,Psikoloji,4
1,S00002,Ogrenci_2,Psikoloji,1
2,S00003,Ogrenci_3,Psikoloji,2
3,S00004,Ogrenci_4,Mimarlik,1
4,S00005,Ogrenci_5,Bilgisayar Muh.,2


# Kaynakların kodlanması kısmına geçiyorum

In [3]:
rooms = pd.DataFrame([
    {"Room": "Arena A", "Capacity": 200, "RequiresPCOnly": False, "ClosedOnFriday": True},
    {"Room": "Arena B", "Capacity": 100, "RequiresPCOnly": False, "ClosedOnFriday": False},
    {"Room": "Salon 1", "Capacity": 50, "RequiresPCOnly": False, "ClosedOnFriday": False},
    {"Room": "Salon 2", "Capacity": 50, "RequiresPCOnly": False, "ClosedOnFriday": False},
    {"Room": "PC Lab", "Capacity": 50, "RequiresPCOnly": True, "ClosedOnFriday": False}
])

rooms

,Room,Capacity,RequiresPCOnly,ClosedOnFriday
0,Arena A,200,False,True
1,Arena B,100,False,False
2,Salon 1,50,False,False
3,Salon 2,50,False,False
4,PC Lab,50,True,False


In [53]:
def generate_slots(start_date="2026-03-02", weeks=4):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    session_times = ["09:00", "13:00", "16:00"]

    slots = []
    work_days_count = 0
    current_day = start

    while work_days_count < weeks * 5:  # 4 hafta * 5 iş günü = 20 gün
        if current_day.weekday() < 5:  # 0=Mon, 4=Fri
            for session in session_times:
                slots.append({
                    "Date": current_day.date(),
                    "Weekday": current_day.strftime("%A"),
                    "Time": session
                })
            work_days_count += 1
        current_day += timedelta(days=1)

    return pd.DataFrame(slots)

slots = generate_slots("2026-03-02", weeks=4)

print("Toplam Slot:", len(slots))
slots.head(9)

Toplam Slot: 60


,Date,Weekday,Time
0,2026-03-02,Monday,09:00
1,2026-03-02,Monday,13:00
2,2026-03-02,Monday,16:00
3,2026-03-03,Tuesday,09:00
4,2026-03-03,Tuesday,13:00
5,2026-03-03,Tuesday,16:00
6,2026-03-04,Wednesday,09:00
7,2026-03-04,Wednesday,13:00
8,2026-03-04,Wednesday,16:00


In [54]:
# Cuma günün kontrolünü sağlıyorum
slots[slots["Weekday"] == "Friday"].head(6)

,Date,Weekday,Time
12,2026-03-06,Friday,09:00
13,2026-03-06,Friday,13:00
14,2026-03-06,Friday,16:00
27,2026-03-13,Friday,09:00
28,2026-03-13,Friday,13:00
29,2026-03-13,Friday,16:00


In [55]:
groups = (
    students
    .groupby(["Major", "Year"])
    .size()
    .reset_index(name="StudentCount")
    .sort_values("StudentCount", ascending=False)
)

groups.head(10)

,Major,Year,StudentCount
15,Mimarlik,4,79
2,Bilgisayar Muh.,3,70
16,Psikoloji,1,67
8,Isletme,1,65
1,Bilgisayar Muh.,2,63
12,Mimarlik,1,63
19,Psikoloji,4,62
4,Endustri Muh.,1,62
5,Endustri Muh.,2,62
10,Isletme,3,61


# Şimdi ise Requires_PC tanımlayacağım çünkü:
Görselde “Requires_PC = Yes olan sınavlar PC Lab’dedir” diyor.
  
Biz veri setinde böyle bir kolon görmüyoruz, o yüzden sunum için anlaşılır bir kural koyacağım:

# Bilgisayar Müh. grupları → Requires_PC = Yes, Diğerleri → No

In [56]:
groups["Requires_PC"] = groups["Major"].eq("Bilgisayar Muh.")
groups.head(10)

,Major,Year,StudentCount,Requires_PC
15,Mimarlik,4,79,False
2,Bilgisayar Muh.,3,70,True
16,Psikoloji,1,67,False
8,Isletme,1,65,False
1,Bilgisayar Muh.,2,63,True
12,Mimarlik,1,63,False
19,Psikoloji,4,62,False
4,Endustri Muh.,1,62,False
5,Endustri Muh.,2,62,False
10,Isletme,3,61,False


# Grup bölmek için fonksiyon yazmamız gerekli
Kapasiteyi asla aşamaması şartını böyle garanti altına alımış olacağım

In [57]:
def split_into_sections(row, rooms_df):
    major = row["Major"]
    year = row["Year"]
    count = int(row["StudentCount"])
    requires_pc = bool(row["Requires_PC"])

    if requires_pc:
        max_cap = int(rooms_df.loc[rooms_df["Room"].eq("PC Lab"), "Capacity"].iloc[0])
    else:
        # PC Lab hariç en büyük kapasite
        max_cap = int(rooms_df.loc[~rooms_df["RequiresPCOnly"], "Capacity"].max())

    n_sections = int(np.ceil(count / max_cap))

    sections = []
    remaining = count
    for s in range(1, n_sections + 1):
        section_size = min(max_cap, remaining)
        remaining -= section_size
        sections.append({
            "Major": major,
            "Year": year,
            "Section": s,
            "SectionSize": section_size,
            "Requires_PC": requires_pc
        })

    return sections

all_sections = []
for _, r in groups.iterrows():
    all_sections.extend(split_into_sections(r, rooms))

sections_df = pd.DataFrame(all_sections)
sections_df.head(20), len(sections_df)

(              Major  Year  Section  SectionSize  Requires_PC
 0          Mimarlik     4        1           79        False
 1   Bilgisayar Muh.     3        1           50         True
 2   Bilgisayar Muh.     3        2           20         True
 3         Psikoloji     1        1           67        False
 4           Isletme     1        1           65        False
 5   Bilgisayar Muh.     2        1           50         True
 6   Bilgisayar Muh.     2        2           13         True
 7          Mimarlik     1        1           63        False
 8         Psikoloji     4        1           62        False
 9     Endustri Muh.     1        1           62        False
 10    Endustri Muh.     2        1           62        False
 11          Isletme     3        1           61        False
 12    Endustri Muh.     3        1           61        False
 13  Bilgisayar Muh.     4        1           50         True
 14  Bilgisayar Muh.     4        2           10         True
 15     

# Hangi algoritmayı kullansam daha karlı çıkarım diye YZ'dan destek aldım
Atama algoritması (basit ve anlaşılır)

Burada “kafa karıştırmayan” şekilde en anlaşılır yöntem:

Greedy (Açgözlü) yaklaşım:

* Büyük section’ları önce yerleştir (zoru önce çöz).

* Slot slot ilerle

* Her section için uygun oda bul:

* PC gerekiyorsa → PC Lab

* Arena A cuma kapalı → cuma günleri Arena A seçme

* Kapasite yetmeli

# Uygun oda seçme aşaması

In [58]:
def find_room_for_section(section, date_obj, rooms_df):
    requires_pc = section["Requires_PC"]
    size = int(section["SectionSize"])
    is_friday = (date_obj.strftime("%A") == "Friday")

    # PC gerekiyorsa sadece PC Lab
    if requires_pc:
        room = rooms_df[rooms_df["Room"].eq("PC Lab")].iloc[0]
        if size <= int(room["Capacity"]):
            return room["Room"]
        return None

    # PC gerekmiyorsa: PC Lab hariç uygun odalar
    candidates = rooms_df[~rooms_df["RequiresPCOnly"]].copy()

    # Cuma ise Arena A kapalı
    if is_friday:
        candidates = candidates[~candidates["Room"].eq("Arena A")]

    # Kapasite filtre
    candidates = candidates[candidates["Capacity"] >= size]

    if candidates.empty:
        return None

    # En küçük yeterli kapasiteyi seç (israfı azaltır)
    candidates = candidates.sort_values("Capacity", ascending=True)
    return candidates.iloc[0]["Room"]

# Yerleştirmeleri Schedule olşturup orada göstereceğim.

Burada dikkat edilmesi gereken ana husus şu:
* Aynı slotta bir odada tek section olur. Ama aynı slotta farklı odalarda farklı section’lar olabilir.

In [59]:
def build_schedule(sections_df, slots_df, rooms_df):
    # Büyükleri önce yerleştir
    to_place = sections_df.sort_values("SectionSize", ascending=False).reset_index(drop=True)

    schedule_rows = []
    used = set()  # (Date, Time, Room)

    slot_idx = 0

    for i, sec in to_place.iterrows():
        placed = False

        while slot_idx < len(slots_df) and not placed:
            slot = slots_df.iloc[slot_idx]
            date_obj = datetime.combine(slot["Date"], datetime.min.time())
            time_str = slot["Time"]

            # Bu slotta uygun oda bul
            room_name = find_room_for_section(sec, date_obj, rooms_df)

            if room_name is None:
                # Bu slotta yer yok demek değil; belki başka slotta olur
                slot_idx += 1
                continue

            key = (slot["Date"], time_str, room_name)
            if key in used:
                # O oda bu slotta dolu -> aynı slotta başka oda denemek için
                # Basitlik adına: bir sonraki slot'a geçiyoruz
                slot_idx += 1
                continue

            # Yerleştir
            used.add(key)
            schedule_rows.append({
                "Major": sec["Major"],
                "Year": sec["Year"],
                "Section": sec["Section"],
                "SectionSize": int(sec["SectionSize"]),
                "Requires_PC": bool(sec["Requires_PC"]),
                "Date": slot["Date"],
                "Weekday": slot["Weekday"],
                "Time": time_str,
                "Room": room_name
            })
            placed = True

        if not placed:
            raise RuntimeError(f"Could not place section: {sec.to_dict()} (No more slots available)")

    return pd.DataFrame(schedule_rows)

schedule = build_schedule(sections_df, slots, rooms)
schedule.head(20)

,Major,Year,Section,SectionSize,Requires_PC,Date,Weekday,Time,Room
0,Mimarlik,4,1,79,False,2026-03-02,Monday,09:00,Arena B
1,Psikoloji,1,1,67,False,2026-03-02,Monday,13:00,Arena B
2,Isletme,1,1,65,False,2026-03-02,Monday,16:00,Arena B
3,Mimarlik,1,1,63,False,2026-03-03,Tuesday,09:00,Arena B
4,Psikoloji,4,1,62,False,2026-03-03,Tuesday,13:00,Arena B
5,Endustri Muh.,2,1,62,False,2026-03-03,Tuesday,16:00,Arena B
6,Endustri Muh.,1,1,62,False,2026-03-04,Wednesday,09:00,Arena B
7,Isletme,3,1,61,False,2026-03-04,Wednesday,13:00,Arena B
8,Endustri Muh.,3,1,61,False,2026-03-04,Wednesday,16:00,Arena B
9,Psikoloji,3,1,59,False,2026-03-05,Thursday,09:00,Arena B


# Şimdi ise kuralların ihlal edilip edilmeme durumunu kontrol edeceğim 
Burada 3 kontrolü yapmam gerekli:
* Kapasite aşımı var mı?
* Arena A cuma kullanılmış mı?
* Requires_PC True olup PC Lab dışında oda atanmış mı?

In [60]:
# 1) Kapasite kontrol
schedule = schedule.merge(rooms[["Room", "Capacity"]], on="Room", how="left")
capacity_violations = schedule[schedule["SectionSize"] > schedule["Capacity"]]

# 2) Arena A cuma kontrol
arenaA_friday = schedule[(schedule["Room"] == "Arena A") & (schedule["Weekday"] == "Friday")]

# 3) PC kuralı kontrol
pc_rule_violations = schedule[(schedule["Requires_PC"] == True) & (schedule["Room"] != "PC Lab")]

print("Capacity violations:", len(capacity_violations))
print("Arena A used on Friday:", len(arenaA_friday))
print("PC rule violations:", len(pc_rule_violations))

capacity_violations.head(), arenaA_friday.head(), pc_rule_violations.head()

Capacity violations: 0
Arena A used on Friday: 0
PC rule violations: 0


(Empty DataFrame
 Columns: [Major, Year, Section, SectionSize, Requires_PC, Date, Weekday, Time, Room, Capacity]
 Index: [],
 Empty DataFrame
 Columns: [Major, Year, Section, SectionSize, Requires_PC, Date, Weekday, Time, Room, Capacity]
 Index: [],
 Empty DataFrame
 Columns: [Major, Year, Section, SectionSize, Requires_PC, Date, Weekday, Time, Room, Capacity]
 Index: [])

In [3]:
# Son kez genel kontrollerini sağlayacağım

In [4]:
import pandas as pd

ROOM_CAPS = {
    "Arena A": 200,
    "Arena B": 100,
    "Salon 1": 50,
    "Salon 2": 50,
    "PC Lab": 50
}

VALID_TIMES = {"09:00", "13:00", "16:00"}
WORKDAYS = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday"}

def validate_schedule(df: pd.DataFrame) -> None:
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df["weekday_calc"] = df["Date"].dt.day_name()

    errors = []

    # 1) Section unique mi?
    dup_sections = df.duplicated(subset=["Major", "Year", "Section"]).sum()
    if dup_sections > 0:
        errors.append(f"❌ Aynı (Major,Year,Section) tekrar etmiş: {dup_sections}")

    # 2) Room-Time çakışması var mı?
    dup_room_time = df.duplicated(subset=["Date", "Time", "Room"]).sum()
    if dup_room_time > 0:
        errors.append(f"❌ Aynı (Date,Time,Room) çakışmış: {dup_room_time}")

    # 3) Weekday doğru mu?
    mismatch = (df["Weekday"] != df["weekday_calc"]).sum()
    if mismatch > 0:
        errors.append(f"❌ Weekday yanlış yazılmış: {mismatch}")

    # 4) Saat valid mi?
    invalid_time = (~df["Time"].isin(VALID_TIMES)).sum()
    if invalid_time > 0:
        errors.append(f"❌ Geçersiz saat var: {invalid_time}")

    # 5) İş günü mü?
    non_workday = (~df["weekday_calc"].isin(WORKDAYS)).sum()
    if non_workday > 0:
        errors.append(f"❌ İş günü olmayan tarihler var: {non_workday}")

    # 6) Arena A cuma kapalı
    arenaA_friday = ((df["Room"] == "Arena A") & (df["weekday_calc"] == "Friday")).sum()
    if arenaA_friday > 0:
        errors.append(f"❌ Arena A cuma kullanılmış: {arenaA_friday}")

    # 7) PC Lab kuralı
    pc_false_in_pclab = ((df["Room"] == "PC Lab") & (df["Requires_PC"] == False)).sum()
    if pc_false_in_pclab > 0:
        errors.append(f"❌ Requires_PC=False ama PC Lab'e konmuş: {pc_false_in_pclab}")

    pc_true_not_in_pclab = ((df["Room"] != "PC Lab") & (df["Requires_PC"] == True)).sum()
    if pc_true_not_in_pclab > 0:
        errors.append(f"❌ Requires_PC=True ama PC Lab dışında: {pc_true_not_in_pclab}")

    # 8) Oda ve kapasite kontrolü
    invalid_rooms = set(df["Room"]) - set(ROOM_CAPS.keys())
    if invalid_rooms:
        errors.append(f"❌ Tanımsız oda var: {invalid_rooms}")

    df["cap_expected"] = df["Room"].map(ROOM_CAPS)

    cap_field_mismatch = (df["Capacity"] != df["cap_expected"]).sum()
    if cap_field_mismatch > 0:
        errors.append(f"❌ Capacity sütunu oda kapasitesiyle uyuşmuyor: {cap_field_mismatch}")

    over_cap = (df["SectionSize"] > df["cap_expected"]).sum()
    if over_cap > 0:
        errors.append(f"❌ Kapasite aşımı var (SectionSize > Room Capacity): {over_cap}")

    if errors:
        print("\n".join(errors))
    else:
        print("✅ Schedule VALID. Kural ihlali yok.")

# Kullanım:
schedule = pd.read_csv("final_schedule.csv")
validate_schedule(schedule)

✅ Schedule VALID. Kural ihlali yok.


# Şimdi ise aldığım çıktıyı daha elverişli olsun diye daha sıralı bir tablo haline getireceğim.

In [5]:
schedule_sorted = schedule.sort_values(["Date", "Time", "Room"]).reset_index(drop=True)
schedule_sorted.head(50)

,Major,Year,Section,SectionSize,Requires_PC,Date,Weekday,Time,Room,Capacity
0,Mimarlik,4,1,79,False,2026-03-02,Monday,09:00,Arena B,100
1,Psikoloji,1,1,67,False,2026-03-02,Monday,13:00,Arena B,100
2,Isletme,1,1,65,False,2026-03-02,Monday,16:00,Arena B,100
3,Mimarlik,1,1,63,False,2026-03-03,Tuesday,09:00,Arena B,100
4,Psikoloji,4,1,62,False,2026-03-03,Tuesday,13:00,Arena B,100
5,Endustri Muh.,2,1,62,False,2026-03-03,Tuesday,16:00,Arena B,100
6,Endustri Muh.,1,1,62,False,2026-03-04,Wednesday,09:00,Arena B,100
7,Isletme,3,1,61,False,2026-03-04,Wednesday,13:00,Arena B,100
8,Endustri Muh.,3,1,61,False,2026-03-04,Wednesday,16:00,Arena B,100
9,Psikoloji,3,1,59,False,2026-03-05,Thursday,09:00,Arena B,100


# Çıktının csv olarak dışa aktarımını yapacağım 

In [6]:
# 1. Exam ID Sütununu Oluştur (Bölüm_Sınıf_Şube)
schedule_sorted['Exam ID'] = schedule_sorted['Major'] + '_' + schedule_sorted['Year'].astype(str) + '_' + schedule_sorted['Section'].astype(str)

# 2. Sadece İstenen Sütunları Filtrele
hedef_tablo = schedule_sorted[['Exam ID', 'Date', 'Time', 'Room']].copy()

# 3. Sütun İsimlerini İstenen Formata (Türkçe) Çevir
hedef_tablo.columns = ['Exam ID', 'Day', 'Time', 'Room']

# 4. Nihai Dosyayı CSV Olarak Çıktı Al
hedef_tablo.to_csv("sinav_programi.csv", index=False)

# Sonucu Notebook'ta Görüntüle
hedef_tablo.head()

,Exam ID,Day,Time,Room
0,Mimarlik_4_1,2026-03-02,09:00,Arena B
1,Psikoloji_1_1,2026-03-02,13:00,Arena B
2,Isletme_1_1,2026-03-02,16:00,Arena B
3,Mimarlik_1_1,2026-03-03,09:00,Arena B
4,Psikoloji_4_1,2026-03-03,13:00,Arena B


## Summary (What / How / Why)

### What did I do?
I created an exam/session schedule by assigning each student group (Major + Year) into time slots and rooms.

### How did I do it?
1. Loaded students data from CSV
2. Built groups (Major + Year)
3. Split groups into smaller sections to never exceed room capacity
4. Generated 60 time slots (4 weeks, Mon-Fri, 3 sessions/day)
5. Assigned each section into a slot and a room using a simple greedy strategy
6. Verified constraints (capacity, Arena A Friday closed, PC Lab rule)

### Why did I do it this way?
- Notebook shows both the code and the results (transparent work)
- Greedy approach is easy to explain and works well for a first solution
- Constraint checks prove the schedule is valid
-------------------------------------------------------------------------------------------------------------------------------------------------------

### Neler Yapıldı?
* Her öğrenci grubunu (Bölüm + Sınıf) belirli zaman dilimlerine ve dersliklere atayarak kapsamlı bir sınav/oturum çizelgesi oluşturulmuştur.

### Yöntem (Nasıl Yapıldı?)
1. Veri Girişi: Öğrenci verileri CSV dosyasından içe aktarıldı.

2. Gruplandırma: Veriler Bölüm ve Sınıf bazında gruplandırıldı.

3. Şubelere Ayırma: Derslik kapasitelerinin aşılmaması adına gruplar daha küçük şubelere bölündü.

4. Zaman Dilimi Oluşturma: 4 hafta boyunca, hafta içi her gün 3 oturum olacak şekilde toplam 60 zaman dilimi üretildi.

5. Atama Algoritması: Her şube, "greedy" (açgözlü) yaklaşımı kullanılarak uygun bir zaman dilimine ve dersliğe yerleştirildi.

6. Kısıt Doğrulama: Belirlenen kısıtların (kapasite uygunluğu, Arena A'nın Cuma günleri kapalı olması ve PC Laboratuvarı kullanım kuralları) ihlal edilmediği doğrulandı.

### Neden Bu Yöntem Tercih Edildi?
* Şeffaflık: Notebook yapısı, kodların ve sonuçların aynı anda görüntülenmesine olanak tanıyarak çalışma sürecini şeffaf kılar.

* Anlaşılabilirlik: "Greedy" yaklaşımı, mantıksal olarak açıklanması kolay ve ilk aşama çözümler için oldukça verimli bir yöntemdir.

* Geçerlilik: Yapılan kısıt kontrolleri, oluşturulan çizelgenin uygulanabilir ve geçerli olduğunu kanıtlar.